# `json2vec` Hello World

This notebook trains a tiny model from an in-memory synthetic dataset.

In [1]:
import lightning.pytorch as lit
import polars as pl
import torch
from rich.pretty import pprint

import json2vec as j2v

In [2]:
records = pl.DataFrame(
    {
        "color": ["red", "orange", "yellow", "blue", "green", "purple"],
        "label": ["warm", "warm", "warm", "cool", "cool", "cool"],
    }
)

In [3]:
params = j2v.Hyperparameters(
    d_model=16,
    fields=j2v.Array(
        name="record",
        fields=[
            j2v.Category(name="color", query="[*].color", max_vocab_size=16),
            j2v.Category(name="label", query="[*].label", max_vocab_size=8, topk=[2]),
        ],
    ),
    target=j2v.Address("record", "label"),
    embed=j2v.Address("record"),
)


model = j2v.Architecture(
    hyperparameters=params,
    batch_size=4,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.PolarsDataModule.from_model(
    model,
    dataframe=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    sample_rate=1.0,
)

2026-05-20 20:01:43.162 | INFO     | json2vec.architecture.root:__init__:182 - initialized JSON2Vec module


In [4]:
trainer = lit.Trainer(
    accelerator="mps",
    max_epochs=20,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    # enable_checkpointing=False,
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/grantham/Desktop/json2vec-oss/examples/checkpoints exists and is not empty.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/j

In [5]:
batch = [[{"color": "red"}], [{"color": "blue"}]]

In [6]:
pprint(model.predict(batch))

{
│   'record/label': {
│   │   'state': {
│   │   │   'valued': [0.9975401759147644, 0.9945204854011536],
│   │   │   'null': [0.001183041138574481, 0.0017848998541012406],
│   │   │   'padded': [0.0004632002965081483, 0.0009843269363045692],
│   │   │   'masked': [0.0004749165673274547, 0.00217420537956059],
│   │   │   'other': [0.0003386603493709117, 0.0005360910436138511]
│   │   },
│   │   'content': {
│   │   │   'value': ['warm', 'cool'],
│   │   │   'probability': [0.9894134402275085, 0.9937065839767456],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'warm', 'probability': 0.9894134402275085},
│   │   │   │   │   {'label': 'cool', 'probability': 0.010586639866232872}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'cool', 'probability': 0.9937065839767456},
│   │   │   │   │   {'label': 'warm', 'probability': 0.006293410900980234}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

In [7]:
pprint(model.embed(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   0.44236961007118225,
│   │   │   │   0.23590423166751862,
│   │   │   │   -0.11971599608659744,
│   │   │   │   0.2313407063484192,
│   │   │   │   0.028536725789308548,
│   │   │   │   -0.06879302859306335,
│   │   │   │   -0.3073069453239441,
│   │   │   │   -0.2327384203672409,
│   │   │   │   -0.3243888020515442,
│   │   │   │   -0.06772670149803162,
│   │   │   │   0.2883271276950836,
│   │   │   │   0.2180250585079193,
│   │   │   │   -0.2068655788898468,
│   │   │   │   -0.4414626359939575,
│   │   │   │   0.0353533960878849,
│   │   │   │   0.21735060214996338
│   │   │   ],
│   │   │   [
│   │   │   │   -0.20824852585792542,
│   │   │   │   0.18301153182983398,
│   │   │   │   0.3817859888076782,
│   │   │   │   -0.12640170753002167,
│   │   │   │   0.09170173853635788,
│   │   │   │   -0.08759921044111252,
│   │   │   │   0.09530768543481827,
│   │   │   │   0.10332568734884262,
│   │   │   │   0.27810096740722656,
│   │   │   │   -0.37245675921440125,
│   │   │   │   -0.3717048764228821,
│   │   │   │   -0.07283228635787964,
│   │   │   │   0.22948962450027466,
│   │   │   │   0.4229709506034851,
│   │   │   │   -0.2672215700149536,
│   │   │   │   -0.2510942816734314
│   │   │   ]
│   │   ]
│   }
}

In [8]:
model.plot(detail=True)

╭─ JSON2Vec ───────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ n_heads: 4                                                                                                           │
│ d_model: 16                                                                                                          │
│ target: [record/label]                                                                                               │
│ embed: [record]                                                                                                      │
│ parameters: 14454                                                                                                    │
│                                                                                                                      │
│   ┏━ record (array) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │
│   ┃ n_heads: 4                                                                                                     ┃ │
│   ┃ attention: mha                                                                                                 ┃ │
│   ┃ max_length: 1                                                                                                  ┃ │
│   ┃ n_outputs: 1                                                                                                   ┃ │
│   ┃ n_linear: 1                                                                                                    ┃ │
│   ┃ n_layers: 1                                                                                                    ┃ │
│   ┃ parameters: 6608                                                                                               ┃ │
│   ┃                                                                                                                ┃ │
│   ┃   ╭─ color (category) ───────────────────────────────────────────────────────────────────────────────────────╮ ┃ │
│   ┃   │ address: record/color                                                                                    │ ┃ │
│   ┃   │ n_heads: 4                                                                                               │ ┃ │
│   ┃   │ query: [*].color                                                                                         │ ┃ │
│   ┃   │ pooling: query                                                                                           │ ┃ │
│   ┃   │ weight: 1.0                                                                                              │ ┃ │
│   ┃   │ n_linear: 1                                                                                              │ ┃ │
│   ┃   │ max_vocab_size: 16                                                                                       │ ┃ │
│   ┃   │ n_bands: 8                                                                                               │ ┃ │
│   ┃   │ p_unavailable: 0.01                                                                                      │ ┃ │
│   ┃   │ topk: []                                                                                                 │ ┃ │
│   ┃   │ parameters: 4055                                                                                         │ ┃ │
│   ┃   │                                                                                                          │ ┃ │
│   ┃   │ state                                                                                                    │ ┃ │
│   ┃   │   vocabulary_size: 6                                                                                     │ ┃ │
│   ┃   │   vocabulary: [yellow, red, orange, purple, blue, green]                                                 │ ┃ │
│   ┃   │                                                                                                          │ ┃ │
│   ┃  

'<!DOCTYPE html>\n<html>\n<head>\n<meta charset="UTF-8">\n<style>\n\nbody {\n    color: #000000;\n    background-color: #ffffff;\n}\n</style>\n</head>\n<body>\n    <pre style="font-family:Menlo,\'DejaVu Sans Mono\',consolas,\'Courier New\',monospace"><code style="font-family:inherit">╭─ JSON2Vec ───────────────────────────────────────────────────────────────────────────────────────────────────────────╮\n│ n_heads: 4                                                                                                           │\n│ d_model: 16                                                                                                          │\n│ target: [record/label]                                                                                               │\n│ embed: [record]                                                                                                      │\n│ parameters: 14454                                                                                      